# 01 — Data Collection


## 1 — Bootstrap


In [ ]:
import sys
import warnings
from pathlib import Path

warnings.filterwarnings('ignore')

CWD = Path.cwd()
ROOT = CWD.parent if CWD.name == 'notebooks' else CWD
SRC = ROOT / 'src'
assert SRC.exists(), f'Could not find src/ at {SRC}'
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

import pandas as pd

import config
from collect import (collect_all_prices, collect_news, collect_prices,
                     save_news, verify_all_raw)

config.ensure_dirs()
pd.set_option('display.width', 140)

print(f'Registered stocks: {config.list_stocks()}')


## 2 — Current state of the raw data


In [ ]:
status = verify_all_raw()
print()
print(status[['stock', 'ok', 'rows', 'start', 'end', 'years',
              'usable_for_horizon_3']].to_string(index=False))


## 3 — Download prices for every registered stock


In [ ]:
# overwrite=False skips stocks whose CSV already exists.
# Set overwrite=True to refresh everything with current data.

paths = collect_all_prices(overwrite=False)
print()
for key, path in paths.items():
    print(f'  {key:12s} {path.name}')


## 4 — Download a single stock, or extend its history


In [ ]:
# Example: re-download Reliance with a longer window.
# Its registry entry starts in 2022, which yields only ~440 usable rows
# and produced no signal at any horizon in Phase 3.

# collect_prices('RELIANCE', start='2014-01-01', end='2024-01-01',
#                overwrite=True)

print('Uncomment the call above to extend a stock\'s history.')


## 5 — Re-verify after downloading


In [ ]:
status = verify_all_raw()
print()
print(status[['stock', 'ok', 'rows', 'start', 'end', 'years',
              'usable_for_horizon_3']].to_string(index=False))
print()
print('Next: run notebooks/00_setup_and_build.ipynb to rebuild datasets.')


## 6 — News collection (optional, live pipeline only)


In [ ]:
# Requires a NewsAPI key in a .env file at the project root:
#     NEWSAPI_KEY=your_key_here
#
# The free tier only serves ~30 days of history, which is far too little
# to train on. This exists for the future live pipeline, not for building
# a training corpus.
#
# The corpus currently in data/raw/ is Reddit r/worldnews (2008-2016),
# NOT company news. Phase 1 measured its ROC-AUC at 0.47 - below chance -
# so sentiment is excluded from the model.

try:
    news = collect_news('TCS', days_back=28)
    display(news.head())
except ValueError as exc:
    print(exc)


## 7 — Save collected news


In [ ]:
# save_news(news, 'TCS')

print('Uncomment to save. This overwrites data/raw/TCS_news_raw.csv.')
